# Modeler - Entrenamiento y evaluacion de modelos de churn

Notebook de modelado del TP3. Recibe las features ya transformadas (`features_train.parquet`, `features_test.parquet`) generadas por el notebook 3 (Training) y produce:

- **Baseline**: `DecisionTreeClassifier` con parametros por defecto
- **DT optimizado**: GridSearchCV sobre DecisionTree
- **Modelo potente**: `RandomForestClassifier` + GridSearchCV
- **Analisis SHAP** sobre Random Forest
- **Evaluacion final** en test set: matriz de confusion, precision, recall, F1, ROC-AUC

El test set solo se toca en la seccion de evaluacion final (seccion 7).

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

FEATURES_TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "features_train.parquet"
FEATURES_TEST_PATH  = PROJECT_ROOT / "data" / "processed" / "features_test.parquet"
TARGET_TRAIN_PATH   = PROJECT_ROOT / "data" / "processed" / "target_train.csv"
TARGET_TEST_PATH    = PROJECT_ROOT / "data" / "processed" / "target_test.csv"
MODELS_OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "models"
MODELS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = ["roc_auc", "f1", "recall", "precision"]

## 1. Ingenieria de features

El pipeline (`src/features/pipeline.py`) ya aplico las transformaciones sobre el split de entrenamiento. Las 33 columnas del parquet son:

| Grupo | Cantidad | Detalle |
|---|---|---|
| Numericas base | 12 | Variables originales del dataset menos `DaySinceLastOrder` |
| Numericas derivadas | 4 | Features de negocio construidas a partir del EDA |
| Categoricas OHE | 17 | 5 variables categoricas expandidas con One-Hot Encoding |

**Features derivadas (decision de negocio):**

- `valor_cliente_proxy` = `OrderCount` * `CashbackAmount` — estima el valor economico del cliente segun volumen y cashback acumulado.
- `coupon_per_order` = `CouponUsed` / `OrderCount` — intensidad de uso de cupones por orden; separa clientes que usan cupones sistematicamente de los ocasionales.
- `cashback_per_order` = `CashbackAmount` / `OrderCount` — cashback promedio por orden; normaliza el beneficio por nivel de actividad.
- `complain_x_satisfaction` = `Complain` * `SatisfactionScore` — interaccion directa de H7: captura el efecto combinado de tener reclamo con nivel de satisfaccion.

`DaySinceLastOrder` fue excluida del modelo (Decision 15): es retroactiva y no sirve como señal de alerta temprana en produccion.

In [3]:
from src.features.pipeline import (
    BASE_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    DERIVED_NUMERIC_FEATURES,
    NUMERIC_FEATURES,
)

X_train = pd.read_parquet(FEATURES_TRAIN_PATH)
X_test  = pd.read_parquet(FEATURES_TEST_PATH)
y_train = pd.read_csv(TARGET_TRAIN_PATH)["Churn"].values
y_test  = pd.read_csv(TARGET_TEST_PATH)["Churn"].values

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train churn rate: {y_train.mean():.3%}")
print(f"y_test  churn rate: {y_test.mean():.3%}")
print()
print("Features numericas base:", BASE_NUMERIC_FEATURES)
print()
print("Features numericas derivadas:", DERIVED_NUMERIC_FEATURES)
print()
print("Features categoricas:", CATEGORICAL_FEATURES)
print()
print(f"Total columnas: {X_train.shape[1]}")


X_train: (4504, 33)
X_test:  (1126, 33)
y_train churn rate: 16.829%
y_test  churn rate: 16.874%

Features numericas base: ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'CashbackAmount']

Features numericas derivadas: ['valor_cliente_proxy', 'coupon_per_order', 'cashback_per_order', 'complain_x_satisfaction']

Features categoricas: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']

Total columnas: 33


## 2. Baseline: DecisionTreeClassifier

Arbol de decision con parametros por defecto y `class_weight='balanced'` para compensar el desbalance (~17% churn). Sirve como piso de comparacion para el modelo potente.

Se evalua con cross-validation estratificada de 5 folds. La metrica principal es ROC-AUC porque accuracy es misleading con clases desbalanceadas (un modelo que predice "nadie churna" alcanza 83% de accuracy).

In [4]:
dt_baseline = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)

cv_baseline = cross_validate(
    dt_baseline, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1
)

print("Decision Tree baseline (CV 5-fold):")
print("-" * 40)
for metric in SCORING:
    scores = cv_baseline[f"test_{metric}"]
    print(f"  {metric:12s}: {scores.mean():.4f} (+/- {scores.std():.4f})")


Decision Tree baseline (CV 5-fold):
----------------------------------------
  roc_auc     : 0.8715 (+/- 0.0253)
  f1          : 0.7791 (+/- 0.0379)
  recall      : 0.7916 (+/- 0.0480)
  precision   : 0.7684 (+/- 0.0399)


## 3. Decision Tree optimizado: GridSearchCV

Busqueda de hiperparametros sobre el espacio definido por la catedra. `scoring='roc_auc'` como criterio de seleccion para mantener consistencia con la metrica principal.

In [5]:
dt_params = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "min_samples_split": [2, 5, 10, 20],
    "class_weight": [None, "balanced"],
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    dt_params,
    cv=CV,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_

print(f"Mejor ROC-AUC (CV): {grid_dt.best_score_:.4f}")
print(f"Mejores parametros: {grid_dt.best_params_}")


Fitting 5 folds for each of 160 candidates, totalling 800 fits
Mejor ROC-AUC (CV): 0.9115
Mejores parametros: {'class_weight': None, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10}


In [6]:
# Visualizacion del arbol optimizado (primeros 3 niveles para legibilidad)
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    best_dt,
    feature_names=list(X_train.columns),
    class_names=["No churn", "Churn"],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
    max_depth=3,
)
plt.title("Decision Tree optimizado (primeros 3 niveles)", fontsize=14)
plt.tight_layout()
plt.savefig(MODELS_OUTPUT_DIR / "dt_tree.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: dt_tree.png")


Guardado: dt_tree.png


## 4. Random Forest con GridSearchCV

Modelo potente. Grilla enfocada en profundidad, tamano de hojas y politica de balanceo. Se mantiene `max_features='sqrt'` (default de RF para clasificacion) para diversificar los arboles.

In [7]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 20, None],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt"],
    "class_weight": ["balanced", "balanced_subsample"],
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_params,
    cv=CV,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

print(f"Mejor ROC-AUC (CV): {grid_rf.best_score_:.4f}")
print(f"Mejores parametros: {grid_rf.best_params_}")


Fitting 5 folds for each of 48 candidates, totalling 240 fits
Mejor ROC-AUC (CV): 0.9791
Mejores parametros: {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}


## 5. Comparacion de modelos

Se comparan los tres modelos usando las mismas metricas CV para un juicio justo. El test set todavia no se toca.

In [8]:
cv_best_dt = cross_validate(best_dt, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)
cv_best_rf = cross_validate(best_rf, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)

comparison = {}
for label, cv_res in [
    ("DT baseline", cv_baseline),
    ("DT optimizado", cv_best_dt),
    ("RF optimizado", cv_best_rf),
]:
    comparison[label] = {
        metric: round(cv_res[f"test_{metric}"].mean(), 4) for metric in SCORING
    }

comparison_df = pd.DataFrame(comparison).T
comparison_df.index.name = "modelo"
print("Comparacion CV 5-fold (medias):")
print(comparison_df.to_string())


Comparacion CV 5-fold (medias):
               roc_auc      f1  recall  precision
modelo                                           
DT baseline     0.8715  0.7791  0.7916     0.7684
DT optimizado   0.9115  0.7483  0.7441     0.7535
RF optimizado   0.9791  0.8692  0.8773     0.8616


## 6. Analisis SHAP (Random Forest final)

SHAP explica el fenomeno de churn, no solo el modelo. Visualizaciones:

- **Summary plot**: distribucion global de impacto por feature
- **Bar plot**: importancia media absoluta (para audiencia no tecnica)
- **Waterfall plot**: desglose de un caso concreto de churn detectado

In [9]:
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

# shap_values es lista [class0_array, class1_array] para clasificacion binaria
sv_churn = shap_values[1] if isinstance(shap_values, list) else shap_values[:, :, 1]
ev_churn = (
    explainer.expected_value[1]
    if isinstance(explainer.expected_value, (list, np.ndarray))
    else explainer.expected_value
)

print(f"SHAP values shape (clase churn): {sv_churn.shape}")
print(f"Expected value (clase churn): {ev_churn:.4f}")


SHAP values shape (clase churn): (1126, 33)
Expected value (clase churn): 0.4994


In [10]:
# Summary plot: dot cloud (distribucion de impacto por feature)
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(sv_churn, X_test, show=False)
plt.tight_layout()
plt.savefig(MODELS_OUTPUT_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: shap_summary.png")


Guardado: shap_summary.png


In [11]:
# Bar plot: importancia media absoluta
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(sv_churn, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(MODELS_OUTPUT_DIR / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: shap_bar.png")


Guardado: shap_bar.png


In [12]:
# Waterfall plot: caso concreto de verdadero positivo
y_pred_rf = best_rf.predict(X_test)
tp_mask = (y_pred_rf == 1) & (y_test == 1)
example_idx = int(np.where(tp_mask)[0][0])

explanation_sample = shap.Explanation(
    values=sv_churn[example_idx],
    base_values=ev_churn,
    data=X_test.values[example_idx],
    feature_names=list(X_test.columns),
)
fig = plt.figure(figsize=(12, 7))
shap.plots.waterfall(explanation_sample, show=False)
plt.tight_layout()
plt.savefig(MODELS_OUTPUT_DIR / "shap_waterfall_example.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Waterfall para test idx={example_idx} (verdadero positivo). Guardado.")


Waterfall para test idx=0 (verdadero positivo). Guardado.


## 7. Evaluacion final en test set

El test set se usa una sola vez aqui. Se evalua el Random Forest optimizado. Se reportan todas las metricas requeridas: precision, recall, F1 y ROC-AUC.

In [13]:
y_pred_final = best_rf.predict(X_test)
y_proba_final = best_rf.predict_proba(X_test)[:, 1]

# Matriz de confusion
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_final,
    ax=ax,
    display_labels=["No churn", "Churn"],
    colorbar=False,
    cmap="Blues",
)
plt.title("Random Forest - Matriz de confusion (test set)")
plt.tight_layout()
plt.savefig(MODELS_OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [14]:
# Metricas por clase
print("Reporte de clasificacion (test set):")
print(classification_report(y_test, y_pred_final, target_names=["No churn", "Churn"]))

roc_auc = roc_auc_score(y_test, y_proba_final)
print(f"ROC-AUC (test): {roc_auc:.4f}")


Reporte de clasificacion (test set):
              precision    recall  f1-score   support

    No churn       0.99      0.98      0.99       936
       Churn       0.91      0.96      0.93       190

    accuracy                           0.98      1126
   macro avg       0.95      0.97      0.96      1126
weighted avg       0.98      0.98      0.98      1126

ROC-AUC (test): 0.9976


In [15]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_final)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"Random Forest (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Aleatorio (AUC = 0.500)")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos (Recall)")
ax.set_title("Curva ROC - Test set")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_OUTPUT_DIR / "roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: roc_curve.png")


Guardado: roc_curve.png


In [16]:
# Comparacion DT optimizado vs RF en test set
y_pred_dt = best_dt.predict(X_test)
y_proba_dt = best_dt.predict_proba(X_test)[:, 1]

from sklearn.metrics import f1_score, precision_score, recall_score

test_results = pd.DataFrame({
    "DT optimizado": {
        "roc_auc": roc_auc_score(y_test, y_proba_dt),
        "f1":       f1_score(y_test, y_pred_dt),
        "recall":   recall_score(y_test, y_pred_dt),
        "precision":precision_score(y_test, y_pred_dt),
    },
    "RF optimizado": {
        "roc_auc": roc_auc,
        "f1":       f1_score(y_test, y_pred_final),
        "recall":   recall_score(y_test, y_pred_final),
        "precision":precision_score(y_test, y_pred_final),
    },
}).T.round(4)

print("Resultados en test set:")
print(test_results.to_string())


Resultados en test set:
               roc_auc      f1  recall  precision
DT optimizado   0.9415  0.7789  0.7789     0.7789
RF optimizado   0.9976  0.9337  0.9632     0.9059


## 8. Guardar modelo final

In [ ]:
import pickle

MODEL_PATH = MODELS_OUTPUT_DIR / "best_rf.pkl"
with open(MODEL_PATH, "wb") as f:
    pickle.dump(best_rf, f)
print(f"Modelo guardado en: {MODEL_PATH.relative_to(PROJECT_ROOT)}")
print(f"Parametros: {best_rf.get_params()}")